# Задачи на доказательство теорем с помощью Z3


## Задача 1. Материальная импликация и исключающее ИЛИ

Докажите следующие тождества пропозиционной логики:

1. Импликация через дизъюнкцию: $p \to q \;=\; \lnot p \lor q$
2. Контрапозиция: $p \to q \;=\; \lnot q \to \lnot p$
3. Самообращение исключающего ИЛИ: $(p \oplus q) \oplus q \;=\; p$
4. Выражение XOR через And/Or/Not: $p \oplus q \;=\; (p \lor q) \land \lnot(p \land q)$


In [1]:
from z3 import *

p = Bool("p")
q = Bool("q")

print("Implies(p, q) == Or(Not(p), q):")
prove(Implies(p, q) == Or(Not(p), q))


Implies(p, q) == Or(Not(p), q):
proved


Проверим контрапозицию: `Implies(p, q) == Implies(Not(q), Not(p))`.

In [2]:
print("Implies(p, q) == Implies(Not(q), Not(p)):")
prove(Implies(p, q) == Implies(Not(q), Not(p)))


Implies(p, q) == Implies(Not(q), Not(p)):
proved


Теперь докажем два свойства исключающего ИЛИ (`Xor`): самообращение (применение XOR с одним и тем же
аргументом дважды возвращает исходное значение) и его выражение через `And`/`Or`/`Not`.

In [3]:
print("Xor(Xor(p, q), q) == p:")
prove(Xor(Xor(p, q), q) == p)

print("Xor(p, q) == And(Or(p, q), Not(And(p, q))):")
prove(Xor(p, q) == And(Or(p, q), Not(And(p, q))))


Xor(Xor(p, q), q) == p:


proved
Xor(p, q) == And(Or(p, q), Not(And(p, q))):
proved


**Итог.** Все четыре тождества доказаны (`proved`) — это тавтологии булевой логики: Z3 перебирает все
комбинации значений `p, q` и убеждается, что отрицание каждой формулы невыполнимо. В частности, `Xor` оказывается
самообратной операцией (`(p ⊕ q) ⊕ q = p`), что и лежит в основе классического трюка обмена двух переменных
без временной ячейки через тройной XOR.

## Задача 2. Свойства модуля и знака

Докажите (или опровергните и объясните контрпример) для целых чисел:

- `Abs(x) >= 0` всегда;
- `Abs(x) >= x`;
- `x * x >= x` — верно ли это для всех целых, или только при определённых ограничениях? Найдите минимальное
  дополнительное условие, при котором утверждение становится доказуемым.


In [4]:
x = Int("x")

def Abs(v):
    return If(v >= 0, v, -v)

print("Abs(x) >= 0:")
prove(Abs(x) >= 0)

print("Abs(x) >= x:")
prove(Abs(x) >= x)


Abs(x) >= 0:


proved
Abs(x) >= x:
proved


In [5]:
print("x * x >= x  (для всех целых):")
prove(x * x >= x)   # неожиданно доказывается!


x * x >= x  (для всех целых):


proved


На первый взгляд кажется, что `x * x >= x` должно опровергаться (как для вещественных чисел, где
`x = 0.5` даёт `0.25 < 0.5`). Но для **целых** чисел это утверждение верно всегда: `x*x - x = x*(x-1)` — произведение
двух последовательных целых чисел. Если `x <= 0`, то `x <= 0` и `x - 1 < 0`, значит произведение `>= 0`. Если `x >= 1`,
то оба множителя `>= 0`. Единственная "опасная зона" (0 < x < 1) для целых чисел просто не существует — там нет ни
одного целого значения. Убедимся, что для вещественных чисел утверждение действительно ложно:

In [6]:
xr = Real("x")
print("Real: x * x >= x:")
prove(xr * xr >= xr)   # ожидаем контрпример x = 1/2


Real: x * x >= x:
counterexample
[x = 1/2]


Таким образом, минимальное дополнительное условие, делающее утверждение доказуемым **над вещественными
числами**, — это как раз `x <= 0 or x >= 1` (эквивалентно "x не лежит строго между 0 и 1"). Более простое достаточное
(но не необходимое) условие — `x >= 1`:

In [7]:
print("Implies(x >= 1, x*x >= x)  над Real:")
prove(Implies(xr >= 1, xr * xr >= xr))

print("Implies(x >= 1, x*x >= x)  над Int (тоже верно, это лишь усиление):")
prove(Implies(x >= 1, x * x >= x))


Implies(x >= 1, x*x >= x)  над Real:
proved
Implies(x >= 1, x*x >= x)  над Int (тоже верно, это лишь усиление):
proved


## Задача 3. Битовый хак: проверка чётности через `&`

Известный трюк: `x % 2 == 0` эквивалентно `(x & 1) == 0` для битовых векторов. Докажите это для 32-битных чисел
с учётом знакового представления (отрицательные числа!). Дополнительно докажите, что `x / 2` (целочисленное деление)
для чётного `x` равно `x >> 1` (арифметический сдвиг) — а для нечётного `x` уже нет, и приведите контрпример.


In [8]:
x = BitVec('x', 32)

print("(x % 2 == 0) == ((x & 1) == 0):")
prove((x % 2 == 0) == ((x & 1) == 0))


(x % 2 == 0) == ((x & 1) == 0):
proved


In [9]:
print("Implies(x % 2 == 0, x / 2 == x >> 1):")
prove(Implies(x % 2 == 0, x / 2 == x >> 1))


Implies(x % 2 == 0, x / 2 == x >> 1):
proved


Для чётных чисел утверждение доказано. Проверим, что для нечётных чисел оно, наоборот,
опровергается — Z3 должен найти контрпример:

In [10]:
print("x / 2 == x >> 1  (без ограничения на чётность):")
prove(x / 2 == x >> 1)


x / 2 == x >> 1  (без ограничения на чётность):
counterexample
[x = 4294967295]


Z3 находит контрпример `x = 4294967295`, что в 32-битном знаковом представлении — это `-1`.
Здесь `/` в Z3 для битовых векторов по умолчанию — это **знаковое** деление (округление к нулю), поэтому
`-1 / 2 == 0`. А `>>` — это **арифметический** сдвиг (сохраняющий знак через заполнение единицами), поэтому
`-1 >> 1 == -1`. Для чётных чисел деление и сдвиг всегда совпадают (округлять нечего), а для отрицательных
нечётных чисел они расходятся: деление округляет к нулю (вверх по модулю), а арифметический сдвиг округляет
к минус бесконечности.

## Задача 4. Теорема Пифагора через координаты

Даны три точки на плоскости: `A(0,0)`, `B(a,0)`, `C(0,b)` (прямой угол при `A` по построению). Докажите с помощью
Z3, что квадрат гипотенузы равен сумме квадратов катетов:

$$AB^2 + AC^2 = BC^2$$


In [11]:
a, b = Reals("a b")

AB2 = a**2
AC2 = b**2
BC2 = (a - 0)**2 + (0 - b)**2

prove(AB2 + AC2 == BC2)


proved


**Усложнение.** Возьмём точку `A` не в начале координат, а произвольную `A(x0, y0)`, и построим `B`, `C`
как *произвольные* точки, подчинив их единственному условию — перпендикулярности векторов `AB` и `AC`
(`AB · AC = 0`). Затем докажем ту же теорему уже не "по построению", а как логическое следствие из условия
перпендикулярности.

In [12]:
x0, y0 = Reals("x0 y0")
bx, by = Reals("bx by")   # координаты B
cx, cy = Reals("cx cy")   # координаты C

AB = (bx - x0, by - y0)
AC = (cx - x0, cy - y0)

perpendicular = AB[0] * AC[0] + AB[1] * AC[1] == 0   # прямой угол при A

AB2 = AB[0]**2 + AB[1]**2
AC2 = AC[0]**2 + AC[1]**2
BC2 = (bx - cx)**2 + (by - cy)**2

prove(Implies(perpendicular, AB2 + AC2 == BC2))


proved


Доказано: как только векторы `AB` и `AC` перпендикулярны (их скалярное произведение равно нулю),
теорема Пифагора выполняется автоматически, независимо от положения точки `A` и конкретных координат `B` и `C`.
Перпендикулярность катетов — необходимое и достаточное условие для равенства `AB² + AC² = BC²`.

## Задача 5. Корректность алгоритма — инвариант цикла суммирования

Рассмотрим простой цикл, вычисляющий сумму чисел от 1 до `n`:

```python
def loop_sum(n, steps):
    s = 0
    i = 0
    for _ in range(steps):
        i = If(i < n, i + 1, i)
        s = If(i <= n, s + i, s)
    return s, i
```

Докажем, что после `n` шагов цикла (`steps == n`, при конкретном малом `n`, например `n = 5`) значение `s`
равно `n*(n+1)/2` — символической формуле для суммы арифметической прогрессии.

In [13]:
def loop_sum(n, steps):
    s = IntVal(0)
    i = IntVal(0)
    for _ in range(steps):
        i = If(i < n, i + 1, i)
        s = If(i <= n, s + i, s)
    return s, i

n = 5
s, i = loop_sum(n, n)
prove(s == n * (n + 1) / 2)


proved


Для конкретного `n = 5` цикл разворачивается в 5 шагов, и Z3 без труда проверяет, что
получившаяся (развёрнутая) формула эквивалентна `n*(n+1)/2`. Но здесь `n` — обычное число Python (5), а не
символ: разворачивание цикла (`for _ in range(steps)`) требует, чтобы число шагов было известно на этапе
построения формулы.

Теперь попробуем доказать то же самое для **произвольного** `n` одним вызовом `prove`, не разворачивая цикл,
а описав рекуррентность через `Function` и аксиомы (`ForAll`):

In [14]:
n_sym = Int('n')
S = Function('S', IntSort(), IntSort())  # S(k) = сумма чисел от 1 до k
k = Int('k')

solver = Solver()
solver.set('timeout', 3000)  # мс, чтобы не зависнуть навсегда

# Рекуррентное определение суммы: S(0) = 0, S(k+1) = S(k) + (k+1)
solver.add(S(0) == 0)
solver.add(ForAll([k], Implies(k >= 0, S(k + 1) == S(k) + (k + 1))))

solver.add(n_sym >= 0)
solver.add(S(n_sym) != n_sym * (n_sym + 1) / 2)   # отрицание цели

result = solver.check()
print("result:", result)   # ожидаем 'unknown', а не 'unsat'


result: unknown


**Почему Z3 не справляется "в лоб".** Аксиомы `S(0) == 0` и `ForAll k. S(k+1) == S(k) + (k+1)`
задают рекуррентность, но сами по себе они не говорят солверу, что нужно "раскрутить" эту рекуррентность для
*произвольного символического* `n`. SMT-солвер работает с конечными инстанциациями кванторов (эвристический
`instantiation` по триггерам): чтобы проверить `S(n) == n(n+1)/2` для конкретного `n`, ему нужно `n`
последовательных применений аксиомы `ForAll`, а `n` здесь — переменная, а не число, поэтому солвер не знает,
сколько раз инстанцировать квантор, и застревает (`unknown`, а не `unsat`/`sat`).

Проблема принципиальная: доказательство `S(n) = n(n+1)/2` для всех `n` — это доказательство **по индукции**
(база `n=0` + индукционный переход `n -> n+1`), а не проверка одной конечной булевой формулы. Голый SMT-солвинг
доказывает только *логическую* (конечную, без квантификации по натуральным числам с содержательной индукцией)
выполнимость/невыполнимость формул — он не выводит универсально квантифицированные факты о рекурсивных
структурах "из общих соображений". Для таких задач нужна отдельная техника: либо доказать индукционный шаг
и базу отдельными (уже конечными!) вызовами `prove`, либо использовать инструменты с встроенной поддержкой
индукции (например, теги `induction` в Dafny/Lean/Coq, или тактику индукции поверх Z3 в специализированных
фреймворках). Продемонстрируем ручное доказательство по индукции — двумя отдельными конечными вызовами
`prove`, которые вместе и составляют полное доказательство:

In [15]:
# База индукции: при n = 0 сумма равна 0
print("База (n=0):")
prove(IntVal(0) == IntVal(0) * (IntVal(0) + 1) / 2)


База (n=0):
proved


In [16]:
# Индукционный переход: если формула верна для k, она верна и для k+1
k = Int('k')
Sk = Int('Sk')  # символическое значение S(k), про которое мы предполагаем гипотезу индукции

hypothesis = Sk == k * (k + 1) / 2          # индукционная гипотеза для k
step_value = Sk + (k + 1)                    # S(k+1) по рекуррентности
goal = step_value == (k + 1) * (k + 2) / 2   # что должно получиться для k+1

print("Индукционный переход (k -> k+1):")
prove(Implies(And(k >= 0, hypothesis), goal))


Индукционный переход (k -> k+1):
proved


Оба вызова `prove` — конечные, чисто алгебраические формулы, и Z3 легко их доказывает.
База + индукционный переход вместе логически эквивалентны исходному утверждению «для всех `n >= 0`:
`S(n) = n(n+1)/2`», но сама индукция как *схема рассуждения* остаётся снаружи Z3 — её выполняет уже человек
(или внешний инструмент), собирая по кусочкам доказательство, которое SMT-солвер не может построить
самостоятельно в одном вызове.